# 6f · Flat schema.org baseline  (whole vocabulary in the prompt, one call, no pipeline)

The naive baseline the pipeline is measured against: put the **entire schema.org vocabulary**
(every class with its parent, every property with its domain/range signature) in the system prompt,
give the model the article, and ask for the graph **directly in schema.org terms** in ONE call.
No retrieval, no shortlist, no minting, no review, no consolidation, no grounding at extraction time.

It answers *"what does the pipeline buy over just prompting?"* on the same model, the same articles
and the same instruments (nb9b grounding precision, nb10 MINE recall), plus one instrument only this
arm needs: **term validity** (the share of emitted types / properties that actually exist in
schema.org), because with nothing enforcing the vocabulary the model can invent a term.

- **Vocabulary:** the SAME pool the pipeline retrieves from — `schema_org_files/schema_classes.json`
  (933 classes) and `schema_properties.json` (1,521 properties) — so neither arm sees a term the other
  cannot.
- **Output:** `results/<RESULTS_MODEL>/<id>/e2e/nary_graph_dedup_<RUN_TAG>_extended_consolidated.json`
  in a SEPARATE results tree (`gemini-2.5-flash-flat`). The file name copies the pipeline's final
  graph name on purpose: nb9b (`GROUND_MODEL=... GROUND_GRAPH=...`) and nb10 (`MINE_MODELS=...
  MINE_TAG=<RUN_TAG>`) then score it without a single edit. The graph carries `type_source` /
  `property_source` (`schema.org` | `invalid`), which is what those notebooks bucket by.
- **Config:** `$FLAT_SPLIT` (val | test), `$FLAT_WORKERS`, `$FLAT_REPEAT`; the Gemini settings copy
  the pipeline's (dynamic thinking, temperature 0, no output cap) so the only difference is the method.


In [1]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [2]:
# ---- CONFIG ---------------------------------------------------------------------------------
import os, sys, json, re, time, pathlib, importlib

BACKEND        = "gemini-2.5-flash"                 # the model doing the work (a BACKENDS key)
RESULTS_MODEL  = os.environ.get("FLAT_RESULTS") or "gemini-2.5-flash-flat"   # results/<this>/
SPLIT          = os.environ.get("FLAT_SPLIT") or "val"                       # val | test
REPEAT         = int(os.environ.get("FLAT_REPEAT") or 1)
WORKERS        = int(os.environ.get("FLAT_WORKERS") or 4)
VARIANT        = "dedup"                            # keeps the file name pattern nb10 derives
RUN_TAG        = f"flat_{SPLIT}_r{REPEAT}"
THINKING_BUDGET   = -1                              # same as the pipeline's 6c: dynamic thinking
MAX_OUTPUT_TOKENS = None                            # same as the pipeline's 6c: no cap
# Stop the batch if the FIRST article costs more than this many total tokens — the estimate the run
# was authorised on is ~25k prompt + ~20k output/thinking per article; 4x that means something is off.
FIRST_ARTICLE_TOKEN_CEILING = 200_000

for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "experiment_config.py").exists():
        sys.path.insert(0, str(_p)); break
import experiment_config as xcfg; importlib.reload(xcfg)

RESULTS_DIR = ROOT / "results" / RESULTS_MODEL
DATASET_DIR = ROOT / "dataset/articles"
SCHEMA_DIR  = ROOT / "schema_org_files"
GLOBAL_DIR  = RESULTS_DIR / "_global" / "e2e"
_osfx       = f"_{VARIANT}_{RUN_TAG}"
OUTPUT_NAME = f"e2e/nary_graph{_osfx}_extended_consolidated.json"   # see the header for why this name

ARTICLE_IDS = xcfg._split_articles({"val": "validation"}.get(SPLIT, SPLIT))
ARTICLE_ID  = ARTICLE_IDS[0]
print(f"split {SPLIT!r}: {len(ARTICLE_IDS)} articles {ARTICLE_IDS[:6]}...")
print(f"-> results/{RESULTS_MODEL}/<id>/{OUTPUT_NAME}")

split 'val': 20 articles ['art1', 'art2', 'art3', 'art4', 'art5', 'art8']...
-> results/gemini-2.5-flash-flat/<id>/e2e/nary_graph_dedup_flat_val_r1_extended_consolidated.json


In [3]:
# --- Gemini client (same call shape as 6c: JSON mime type, dynamic thinking, temperature 0) -------
from google import genai
from google.genai import types as genai_types
try:
    from json_repair import repair_json
except Exception:
    repair_json = None

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in the env."
client = genai.Client(api_key=GEMINI_API_KEY)

for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend
importlib.reload(nb_parallel); importlib.reload(llm_backend)
nb_parallel.install()

def _loads_lenient(text, resp=None):
    t = (text or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\n?", "", t); t = re.sub(r"\n?```$", "", t).strip()
    for cand in (t, re.sub(r",(\s*[}\]])", r"\1", t)):
        try:
            return json.loads(cand)
        except json.JSONDecodeError:
            pass
    if repair_json is not None:
        try:
            obj = json.loads(repair_json(t))
            print(f"    [json_repair] salvaged a malformed response ({len(t)} chars)")
            return obj
        except Exception:
            pass
    fr = None
    try: fr = resp.candidates[0].finish_reason
    except Exception: pass
    (RESULTS_DIR / "_debug_last_bad_response.json").write_text(t)
    raise ValueError(f"Gemini JSON unparseable (finish_reason={fr}; {len(t)} chars)")

_RETRY_STATUS = (429, 500, 502, 503, 504)
_RETRY_WORDS  = ("unavailable", "resource_exhausted", "internal", "high demand", "overloaded",
                 "timed out", "timeout", "connection", "server disconnected")
_FATAL_WORDS  = ("credits are depleted", "billing", "quota exceeded for quota metric",
                 "exceeded your current quota", "insufficient")

def _retryable(exc):
    msg = str(exc).lower()
    if any(w in msg for w in _FATAL_WORDS):
        return False
    code = getattr(exc, "code", None) or getattr(exc, "status_code", None)
    return code in _RETRY_STATUS or any(w in msg for w in _RETRY_WORDS) \
        or type(exc).__name__ in ("ConnectError", "ReadTimeout", "RemoteProtocolError", "ConnectionError")

def _call_with_retry(fn, tries=5, backoff=10):
    for i in range(tries):
        try:
            return fn()
        except Exception as exc:                       # noqa: BLE001
            if i == tries - 1 or not _retryable(exc):
                raise
            wait = backoff * (2 ** i)
            print(f"    retry {i + 1}/{tries - 1} in {wait}s: {type(exc).__name__}: {str(exc)[:160]}")
            time.sleep(wait)

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    cfg = dict(system_instruction=system_prompt, temperature=temperature,
               response_mime_type="application/json")
    if MAX_OUTPUT_TOKENS is not None:
        cfg["max_output_tokens"] = MAX_OUTPUT_TOKENS
    if THINKING_BUDGET is not None:
        cfg["thinking_config"] = genai_types.ThinkingConfig(thinking_budget=THINKING_BUDGET)
    t0 = time.perf_counter()
    resp = _call_with_retry(lambda: client.models.generate_content(
        model=model or GEMINI_MODEL, contents=user_message,
        config=genai_types.GenerateContentConfig(**cfg)))
    nb_parallel.record_gemini_usage(resp, time.perf_counter() - t0)
    return _loads_lenient(resp.text, resp)

# Binds GEMINI_MODEL and RESULTS_DIR; for a gemini backend it leaves gemini_json above alone.
llm_backend.activate(BACKEND, ROOT, globals(), results=RESULTS_MODEL)

backend: GEMINI gemini-2.5-flash  ->  results/gemini-2.5-flash-flat/   (reading/writing another model's tree)


{'backend': 'gemini-2.5-flash',
 'kind': 'gemini',
 'model': 'gemini-2.5-flash',
 'results_dir': PosixPath('/Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/gemini-2.5-flash-flat')}

## 1. The vocabulary — every schema.org class and property, verbalised once

In [4]:
# The pipeline's own parsed pools (nb1 / nb5 outputs), so both arms draw on the identical term set.
CLASSES    = json.loads((SCHEMA_DIR / "schema_classes.json").read_text())
PROPERTIES = json.loads((SCHEMA_DIR / "schema_properties.json").read_text())

CLASS_LABELS = {c["label"] for c in CLASSES}
PROP_LABELS  = {p["label"] for p in PROPERTIES}
_class_ci = {l.casefold(): l for l in CLASS_LABELS}
_prop_ci  = {l.casefold(): l for l in PROP_LABELS}

def _short(x): return x.split(":")[-1]

class_lines = []
for c in sorted(CLASSES, key=lambda c: c["label"]):
    # nb1 stores the ancestor chain as "Thing > Place > AdministrativeArea"; the direct parent is its tail
    parent = (c.get("path") or "").split(" > ")[-1]
    class_lines.append(f"{c['label']}" + (f" < {parent}" if parent else ""))
prop_lines = []
for p in sorted(PROPERTIES, key=lambda p: p["label"]):
    dom = "|".join(p.get("domain") or []) or "Thing"
    rng = "|".join(p.get("range") or []) or "Thing"
    prop_lines.append(f"{p['label']}: {dom} -> {rng}")

VOCAB_TEXT = ("CLASSES (label < parent classes):\n" + "\n".join(class_lines)
              + "\n\nPROPERTIES (label: domain classes -> range classes):\n" + "\n".join(prop_lines))
print(f"{len(CLASSES)} classes, {len(PROPERTIES)} properties -> {len(VOCAB_TEXT):,} chars")
print(class_lines[:3], prop_lines[:3])

933 classes, 1521 properties -> 105,437 chars
['3DModel < MediaObject', 'AMRadioChannel < RadioChannel', 'APIReference < TechArticle'] ['about: Certification|CommunicateAction|CreativeWork|DefinedTerm|DefinedTermSet|Event -> Thing', 'abridged: Book -> Boolean', 'abstract: CreativeWork -> Text']


## 2. The prompt — the bare task

One sentence of task, one JSON skeleton, then the vocabulary. No rules about coreference, evidence,
events, dates or coverage: everything the pipeline decomposes is left to the model in one shot.
The graph therefore has entities and relations only (no n-ary nodes, no evidence spans); nb10 reads
such a graph as is, and nb9b's full-article judge records `evidence_status: none` and still judges
every assertion against the text.

In [5]:
FLAT_SYSTEM_PROMPT = """Given the TEXT and the schema.org vocabulary below, extract a typed knowledge graph of the entities the TEXT mentions and the relations between them. Type each entity with a schema.org class and express each relation with a schema.org property, using only the labels listed below.

Return JSON:
{"entities": [{"id": "E1", "text": "...", "type": "<schema.org class>"}],
 "relations": [{"subject_id": "E1", "property": "<schema.org property>", "object_id": "E2"}]}

""" + VOCAB_TEXT

print(f"system prompt: {len(FLAT_SYSTEM_PROMPT):,} chars")
_ct = client.models.count_tokens(model=GEMINI_MODEL, contents=FLAT_SYSTEM_PROMPT)
PROMPT_TOKENS = _ct.total_tokens
print(f"system prompt: {PROMPT_TOKENS:,} tokens (counted by the API, free)")

system prompt: 105,901 chars
system prompt: 23,071 tokens (counted by the API, free)


## 3. Extraction + deterministic validation

No LLM post-processing. After the call: resolve endpoint texts, drop relations whose endpoints do
not exist, drop duplicate edges, and stamp every type / property with `schema.org` when it is a
listed label, `invalid` when it is not (a case-only mismatch is recorded separately, so it can be
reported as a lenient figure too).

In [6]:
import unicodedata
from collections import Counter

def _demojibake(s):
    if isinstance(s, str) and any(m in s for m in ("Ã", "Â", "â€")):
        try: return s.encode("latin-1").decode("utf-8")
        except (UnicodeDecodeError, UnicodeEncodeError): return s
    return s

def _reference_date(text):
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m: return None
    d, mth, y = m.groups(); return f"{y}-{mth}-{d}"

def _check(label, exact, ci):
    """-> (source, canonical_label). exact hit: schema.org; case-only hit: case_mismatch; else invalid."""
    if label in exact:
        return "schema.org", label
    hit = ci.get((label or "").casefold())
    if hit:
        return "case_mismatch", hit
    return "invalid", label

def _norm(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", s or "")).strip().casefold()

def validate_graph(graph):
    """Stamp sources; guard endpoints; dedup edges. Returns (graph, stats)."""
    st = Counter()
    for kind, items in (("entity", graph["entities"]), ("node", graph["nary_nodes"])):
        for it in items:
            src, canon = _check(it.get("type"), CLASS_LABELS, _class_ci)
            it["type_oie"] = it.get("type")
            it["type_source"] = "schema.org" if src == "schema.org" else "invalid"
            it["type_check"] = src                      # schema.org | case_mismatch | invalid
            it["type_vocab_id"] = None; it["type_is_new_to_vocab"] = False
            st[f"{kind}_type_{src}"] += 1
    ids = {e["id"] for e in graph["entities"]} | {n["id"] for n in graph["nary_nodes"]}
    ref = {e["id"]: (e.get("text"), e.get("type")) for e in graph["entities"]}
    ref.update({n["id"]: (n.get("label"), n.get("type")) for n in graph["nary_nodes"]})
    kept, seen = [], set()
    for r in graph["relations"]:
        prop = r.get("property") or r.get("relation_phrase")
        s, o = r.get("subject_id"), r.get("object_id")
        if s not in ids or o not in ids:
            st["relation_dangling_dropped"] += 1; continue
        if s == o:
            st["relation_selfloop_dropped"] += 1; continue
        key = (s, _norm(prop), o)
        if key in seen:
            st["relation_duplicate_dropped"] += 1; continue
        seen.add(key)
        src, canon = _check(prop, PROP_LABELS, _prop_ci)
        r["property"] = prop; r["relation_phrases"] = [prop]
        r["property_source"] = "schema.org" if src == "schema.org" else "invalid"
        r["property_check"] = src
        r["vocab_id"] = None; r["property_is_new_to_vocab"] = False
        r["subject_text"], r["subject_type"] = ref[s]
        r["object_text"],  r["object_type"]  = ref[o]
        if not isinstance(r.get("evidence"), list):
            r["evidence"] = [r["evidence"]] if r.get("evidence") else []
        st[f"relation_property_{src}"] += 1
        kept.append(r)
    graph["relations"] = kept
    return graph, dict(st)

def extract_flat(aid, write=True):
    t0 = time.perf_counter()
    text = _demojibake((DATASET_DIR / f"{aid}.txt").read_text())
    ref_date = _reference_date(text)
    out = gemini_json(FLAT_SYSTEM_PROMPT, f"REFERENCE_DATE: {ref_date}\n\nTEXT:\n{text}")
    usage = nb_parallel.last_usage()
    graph = {"article_id": aid, "reference_date": ref_date, "method": "flat_schema_single_call",
             "backend": BACKEND, "run_tag": RUN_TAG,
             "entities": out.get("entities", []) or [],
             "nary_nodes": out.get("nary_nodes", []) or [],
             "relations": out.get("relations", []) or []}
    for e in graph["entities"]:
        e.setdefault("aliases", [])
    for n in graph["nary_nodes"]:
        if not isinstance(n.get("evidence"), list):
            n["evidence"] = [n["evidence"]] if n.get("evidence") else []
    graph, vstats = validate_graph(graph)
    graph["validation"] = vstats
    graph["consolidated"] = False
    graph["token_usage"] = {"extraction": usage, "review": None, "combined": usage}
    if write:
        (RESULTS_DIR / aid / OUTPUT_NAME).write_text(json.dumps(graph, indent=2, ensure_ascii=False))
    inv_t = vstats.get("entity_type_invalid", 0) + vstats.get("node_type_invalid", 0)
    inv_p = vstats.get("relation_property_invalid", 0)
    print(f"  {aid}: {len(graph['entities'])} entities, {len(graph['nary_nodes'])} nodes, "
          f"{len(graph['relations'])} relations | invalid types {inv_t}, invalid props {inv_p} | "
          f"tokens prompt {usage.get('prompt_tokens')}, out {usage.get('output_tokens')}, "
          f"thoughts {usage.get('thoughts_tokens')} | {time.perf_counter() - t0:.0f}s")
    return graph

## 4. Batch — first article alone (cost check), then the rest in parallel

In [7]:
nb_parallel.reset_usage()
_t_batch = time.perf_counter()
_first = nb_parallel.pmap(extract_flat, ARTICLE_IDS[:1], workers=1)
assert _first[0].ok, f"first article failed: {_first[0].error}"
_u0 = _first[0].value["token_usage"]["combined"]
assert (_u0.get("total_tokens") or 0) <= FIRST_ARTICLE_TOKEN_CEILING, (
    f"first article spent {_u0.get('total_tokens')} tokens > ceiling {FIRST_ARTICLE_TOKEN_CEILING}; "
    f"stopping before the rest of the split. Raise FIRST_ARTICLE_TOKEN_CEILING if this is intended.")
_rest = nb_parallel.pmap(extract_flat, ARTICLE_IDS[1:], workers=WORKERS)
WALL_SECONDS = time.perf_counter() - _t_batch
_res = list(_first) + list(_rest)
GRAPHS = {r.item: r.value for r in _res if r.ok}
FAILED = [r.item for r in _res if not r.ok]
print(f"\n{len(GRAPHS)}/{len(ARTICLE_IDS)} articles written" + (f"; FAILED: {FAILED}" if FAILED else ""))
nb_parallel.usage_report("6f · flat schema.org baseline", wall_seconds=WALL_SECONDS)

1 item(s), running sequentially

[1/1] art1
  art1: 69 entities, 0 nodes, 77 relations | invalid types 0, invalid props 0 | tokens prompt 23983, out 5943, thoughts 14538 | 79s

done — 1/1 ok in 78.9s
19 item(s), running on 4 threads

[1/19] art4  (38.8s)
     art4: 31 entities, 0 nodes, 26 relations | invalid types 0, invalid props 0 | tokens prompt 23529, out 3606, thoughts 6255 | 39s
[2/19] art5  (44.2s)
     art5: 45 entities, 0 nodes, 51 relations | invalid types 0, invalid props 14 | tokens prompt 23942, out 4434, thoughts 6664 | 44s
[3/19] art8  (59.6s)
     art8: 52 entities, 0 nodes, 39 relations | invalid types 0, invalid props 0 | tokens prompt 24406, out 3287, thoughts 11386 | 60s
[4/19] art3  (109.0s)
     art3: 112 entities, 0 nodes, 110 relations | invalid types 0, invalid props 50 | tokens prompt 25388, out 8459, thoughts 21164 | 109s
[5/19] art17  (56.4s)
     art17: 27 entities, 0 nodes, 38 relations | invalid types 1, invalid props 4 | tokens prompt 23714, out 3183, t

{'label': '6f · flat schema.org baseline',
 'totals': {'calls': 20,
  'prompt_tokens': 482049,
  'output_tokens': 88924,
  'thoughts_tokens': 392403,
  'total_tokens': 963376,
  'seconds': 1799.8},
 'wall_seconds': 533.2710593749944,
 'per_tag': {'art1': {'calls': 1,
   'seconds': 78.9,
   'prompt_tokens': 23983,
   'output_tokens': 5943,
   'thoughts_tokens': 14538,
   'total_tokens': 44464},
  'art4': {'calls': 1,
   'seconds': 38.8,
   'prompt_tokens': 23529,
   'output_tokens': 3606,
   'thoughts_tokens': 6255,
   'total_tokens': 33390},
  'art5': {'calls': 1,
   'seconds': 44.2,
   'prompt_tokens': 23942,
   'output_tokens': 4434,
   'thoughts_tokens': 6664,
   'total_tokens': 35040},
  'art8': {'calls': 1,
   'seconds': 59.6,
   'prompt_tokens': 24406,
   'output_tokens': 3287,
   'thoughts_tokens': 11386,
   'total_tokens': 39079},
  'art3': {'calls': 1,
   'seconds': 109.0,
   'prompt_tokens': 25388,
   'output_tokens': 8459,
   'thoughts_tokens': 21164,
   'total_tokens': 5501

## 5. Metrics — the counts the run writes about itself

Graph size, term validity (strict and case-lenient), the share of types / triples on a valid
schema.org term, vocabulary size (distinct valid types / properties across the split), and cost.
Written to `_global/e2e/metrics_flat_<tag>.json` and `token_usage_<tag>.json`, next to where the
pipeline's own stages write theirs.

In [8]:
def _sum(key): return sum(g["validation"].get(key, 0) for g in GRAPHS.values())
def _pct(n, d): return round(100.0 * n / d, 1) if d else None

per_article = []
for aid in ARTICLE_IDS:
    g = GRAPHS.get(aid)
    if not g: continue
    v, u = g["validation"], g["token_usage"]["combined"] or {}
    n_t = len(g["entities"]) + len(g["nary_nodes"])
    n_r = len(g["relations"])
    per_article.append({
        "article": aid, "entities": len(g["entities"]), "nary_nodes": len(g["nary_nodes"]),
        "relations": n_r,
        "types_schema": v.get("entity_type_schema.org", 0) + v.get("node_type_schema.org", 0),
        "types_case_mismatch": v.get("entity_type_case_mismatch", 0) + v.get("node_type_case_mismatch", 0),
        "types_invalid": v.get("entity_type_invalid", 0) + v.get("node_type_invalid", 0),
        "props_schema": v.get("relation_property_schema.org", 0),
        "props_case_mismatch": v.get("relation_property_case_mismatch", 0),
        "props_invalid": v.get("relation_property_invalid", 0),
        "dangling_dropped": v.get("relation_dangling_dropped", 0),
        "duplicate_dropped": v.get("relation_duplicate_dropped", 0),
        "prompt_tokens": u.get("prompt_tokens"), "output_tokens": u.get("output_tokens"),
        "thoughts_tokens": u.get("thoughts_tokens"), "total_tokens": u.get("total_tokens"),
    })

T = {k: sum(r[k] or 0 for r in per_article) for k in per_article[0] if k != "article"}
n_types = T["types_schema"] + T["types_case_mismatch"] + T["types_invalid"]
n_props = T["props_schema"] + T["props_case_mismatch"] + T["props_invalid"]

type_vocab = Counter(); prop_vocab = Counter(); bad_types = Counter(); bad_props = Counter()
for g in GRAPHS.values():
    for it in [*g["entities"], *g["nary_nodes"]]:
        (type_vocab if it["type_source"] == "schema.org" else bad_types)[it.get("type")] += 1
    for r in g["relations"]:
        (prop_vocab if r["property_source"] == "schema.org" else bad_props)[r.get("property")] += 1

METRICS = {
    "stage": "6f_flat_schema_baseline", "backend": BACKEND, "model": GEMINI_MODEL,
    "results_model": RESULTS_MODEL, "split": SPLIT, "run_tag": RUN_TAG, "graph": OUTPUT_NAME,
    "thinking_budget": THINKING_BUDGET, "max_output_tokens": MAX_OUTPUT_TOKENS,
    "vocabulary": {"classes": len(CLASSES), "properties": len(PROPERTIES),
                   "system_prompt_chars": len(FLAT_SYSTEM_PROMPT), "system_prompt_tokens": PROMPT_TOKENS},
    "articles": len(per_article), "failed": FAILED,
    "totals": {"entities": T["entities"], "nary_nodes": T["nary_nodes"], "relations": T["relations"],
               "type_decisions": n_types, "property_decisions": n_props},
    "term_validity": {
        "types_valid_pct_strict":   _pct(T["types_schema"], n_types),
        "types_valid_pct_lenient":  _pct(T["types_schema"] + T["types_case_mismatch"], n_types),
        "props_valid_pct_strict":   _pct(T["props_schema"], n_props),
        "props_valid_pct_lenient":  _pct(T["props_schema"] + T["props_case_mismatch"], n_props),
        "triples_on_schema_pct":    _pct(T["props_schema"], T["relations"]),
        "invalid_types": dict(bad_types.most_common()), "invalid_properties": dict(bad_props.most_common()),
    },
    "vocab_size": {"distinct_valid_types": len(type_vocab), "distinct_valid_properties": len(prop_vocab),
                   "top_types": dict(type_vocab.most_common(25)), "top_properties": dict(prop_vocab.most_common(25))},
    "hygiene": {"dangling_relations_dropped": T["dangling_dropped"], "duplicate_relations_dropped": T["duplicate_dropped"]},
    "cost": {"prompt_tokens": T["prompt_tokens"], "output_tokens": T["output_tokens"],
             "thoughts_tokens": T["thoughts_tokens"], "total_tokens": T["total_tokens"],
             "per_article_total": round(T["total_tokens"] / len(per_article)), "wall_seconds": round(WALL_SECONDS, 1)},
    "per_article": per_article,
}
(GLOBAL_DIR / f"metrics_flat{_osfx}.json").write_text(json.dumps(METRICS, indent=2, ensure_ascii=False))
(GLOBAL_DIR / f"token_usage{_osfx}.json").write_text(json.dumps(
    {"method": "flat_schema_single_call", "model": GEMINI_MODEL, "thinking_budget": THINKING_BUDGET,
     "split": SPLIT, "run_tag": RUN_TAG, "system_prompt_tokens": PROMPT_TOKENS,
     "per_article": [{"article": r["article"], **{k: r[k] for k in ("prompt_tokens", "output_tokens", "thoughts_tokens", "total_tokens")}} for r in per_article],
     "totals": {k: T[k] for k in ("prompt_tokens", "output_tokens", "thoughts_tokens", "total_tokens")}},
    indent=2))

print(json.dumps({k: v for k, v in METRICS.items() if k not in ("per_article",)}, indent=1, ensure_ascii=False)[:6000])
print(f"\nwrote {GLOBAL_DIR / f'metrics_flat{_osfx}.json'}")

{
 "stage": "6f_flat_schema_baseline",
 "backend": "gemini-2.5-flash",
 "model": "gemini-2.5-flash",
 "results_model": "gemini-2.5-flash-flat",
 "split": "val",
 "run_tag": "flat_val_r1",
 "graph": "e2e/nary_graph_dedup_flat_val_r1_extended_consolidated.json",
 "thinking_budget": -1,
 "max_output_tokens": null,
 "vocabulary": {
  "classes": 933,
  "properties": 1521,
  "system_prompt_chars": 105901,
  "system_prompt_tokens": 23071
 },
 "articles": 20,
 "failed": [],
 "totals": {
  "entities": 1175,
  "nary_nodes": 0,
  "relations": 1012,
  "type_decisions": 1175,
  "property_decisions": 1012
 },
 "term_validity": {
  "types_valid_pct_strict": 99.4,
  "types_valid_pct_lenient": 99.4,
  "props_valid_pct_strict": 84.4,
  "props_valid_pct_lenient": 84.4,
  "triples_on_schema_pct": 84.4,
  "invalid_types": {
   "TheftAction": 6,
   "Award": 1
  },
  "invalid_properties": {
   "includes": 20,
   "uses": 9,
   "decodes": 9,
   "isA": 7,
   "performs": 7,
   "creates": 7,
   "partOf": 6,
   "t

## 6. How to score it with the pipeline's instruments (cluster, free)

```bash
# grounding precision (judge: qwen3-32b-think over the vLLM tunnel)
GROUND_MODEL=gemini-2.5-flash-flat GROUND_TAG=flat_val_r1 \
GROUND_GRAPH=nary_graph_dedup_flat_val_r1_extended_consolidated.json  jupyter nbconvert --execute 0_shared/9b_grounding_fulltext.ipynb
# MINE recall (judge: llama33-70b)
MINE_MODELS=gemini-2.5-flash-flat MINE_TAG=flat_val_r1  jupyter nbconvert --execute evaluation/10_mine_benchmark.ipynb
```
Both resolve the graph by the `<model>/<id>/e2e/nary_graph_dedup_<tag>_extended_consolidated.json`
pattern, which is why this notebook writes under that name.